In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
stations_df = all_sheets['stations']
chargers_df = all_sheets['chargers']
sessions_df = all_sheets['sessions']

print(chargers_df.shape)
print(sessions_df.shape)

(199, 5)
(294024, 15)


**Calculate total usage time per charger**

In [ ]:
sessions_df['session_duration_minutes'] = sessions_df['session_duration_minutes'].astype(float)

charger_usage = sessions_df.groupby('charger_id')['session_duration_minutes'].sum().reset_index()
charger_usage.columns = ['charger_id', 'total_minutes_used']

print(charger_usage.head())
print(charger_usage.shape)

      charger_id  total_minutes_used
0  STN_001_CH_01            184859.0
1  STN_001_CH_02             90590.0
2  STN_001_CH_03             94833.0
3  STN_001_CH_04            190348.0
4  STN_002_CH_01            152559.0
(199, 2)


In [ ]:
charger_usage = charger_usage.merge(chargers_df, on='charger_id')
print(charger_usage.head())

      charger_id  total_minutes_used station_id charger_type  max_power_kW  \
0  STN_001_CH_01            184859.0    STN_001      DC_50kW            50   
1  STN_001_CH_02             90590.0    STN_001     DC_150kW           150   
2  STN_001_CH_03             94833.0    STN_001     DC_150kW           150   
3  STN_001_CH_04            190348.0    STN_001      DC_50kW            50   
4  STN_002_CH_01            152559.0    STN_002      DC_50kW            50   

  installation_date  
0        2022-06-07  
1        2022-12-22  
2        2022-10-21  
3        2022-12-08  
4        2022-07-09  


In [ ]:
total_days = (sessions_df['start_timestamp'].max() - sessions_df['start_timestamp'].min()).days
print(total_days)

1095


In [ ]:
total_available_minutes = total_days * 24 * 60
print(total_available_minutes)

1576800


**calculate utilisation percentage for each charger**

In [ ]:
charger_usage['utilisation_pct'] = (charger_usage['total_minutes_used'] / total_available_minutes) * 100
print(charger_usage[['charger_id', 'total_minutes_used', 'utilisation_pct']].head())

      charger_id  total_minutes_used  utilisation_pct
0  STN_001_CH_01            184859.0        11.723681
1  STN_001_CH_02             90590.0         5.745180
2  STN_001_CH_03             94833.0         6.014269
3  STN_001_CH_04            190348.0        12.071791
4  STN_002_CH_01            152559.0         9.675228


In [9]:
charger_usage['idle_pct'] = 100 - charger_usage['utilisation_pct']
print(charger_usage[['charger_id', 'utilisation_pct', 'idle_pct']].head())

      charger_id  utilisation_pct   idle_pct
0  STN_001_CH_01        11.723681  88.276319
1  STN_001_CH_02         5.745180  94.254820
2  STN_001_CH_03         6.014269  93.985731
3  STN_001_CH_04        12.071791  87.928209
4  STN_002_CH_01         9.675228  90.324772


In [10]:
print("TOP 5 MOST-USED CHARGERS (potential bottlenecks):")
print(charger_usage.sort_values('utilisation_pct', ascending=False)[['charger_id', 'station_id', 'utilisation_pct']].head())

print("\nBOTTOM 5 LEAST-USED CHARGERS (underperforming):")
print(charger_usage.sort_values('utilisation_pct', ascending=True)[['charger_id', 'station_id', 'utilisation_pct']].head())

TOP 5 MOST-USED CHARGERS (potential bottlenecks):
       charger_id station_id  utilisation_pct
3   STN_001_CH_04    STN_001        12.071791
35  STN_007_CH_02    STN_007        12.013889
37  STN_007_CH_04    STN_007        11.989536
0   STN_001_CH_01    STN_001        11.723681
36  STN_007_CH_03    STN_007        11.594939

BOTTOM 5 LEAST-USED CHARGERS (underperforming):
        charger_id station_id  utilisation_pct
147  STN_025_CH_02    STN_025         0.733067
165  STN_027_CH_04    STN_027         0.734970
131  STN_023_CH_01    STN_023         0.767567
148  STN_025_CH_03    STN_025         0.778919
167  STN_027_CH_06    STN_027         0.811580


In [11]:
low_usage_stations = ['STN_023', 'STN_025', 'STN_027']
print(stations_df[stations_df['station_id'].isin(low_usage_stations)])

   station_id      station_name      region  activation_year  n_chargers
22    STN_023  EV FastCharge 23    Scotland             2024           8
24    STN_025  EV FastCharge 25  North West             2024           8
26    STN_027  EV FastCharge 27  South East             2024           8


In [12]:
charger_usage.to_csv('/content/drive/MyDrive/Cadetx /charger_usage.csv', index=False)
print("Saved!")

Saved!
